# Ensemble Methods


In [13]:
import pandas as pd

import numpy as np

## Choose your own Regression

**Working together with your partner**, choose an interesting data set to fit *regression* models for, that is, a data set with a *quantitative* target variable and several feature variables. Could be a data set that: we have used in class, we have used on an assignment, a data set from your project, you used in another course, or you just find interesting! You'll work together with your partner to develop *regression* models to predict the *quantitative* target (so you need to decide on the data set and target variable together with your partner).

After you have decided on the data set and target variable:

- One partner should fit a linear regression model
- One partner should fit a k-nearest neighbors regression model

If you're using an example from earlier in the class, you can use a model you have already come up with or you can try something new. Note: you do NOT have to agree on the features included in your models.

After each partner has their model, **work together to combine your code and create two more models**:

- An ensemble model that uses **voting**
- An ensemble model that uses **stacking**

For each of the four models (linear regression, knn regression, voting ensemble, stacking ensemble)

1. Fit the model using the training data
1. Use the fitted model to predict the target for a few observations of your choice, and compare the predictions
1. Estimate the test RMSE using cross-validation, and compare the results

Which model seems to work best?


In [14]:
import pandas as pd
import numpy as np

df_housing = pd.read_csv("http://dlsun.github.io/pods/data/AmesHousing.txt", sep="\t")

predictors = ["Gr Liv Area", "Bedroom AbvGr", "Full Bath"]
X_train = df_housing[predictors]
y_train = df_housing["SalePrice"]

df_housing.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [15]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
linear_model.fit(X=X_train, y=y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](3,)","[ 118.77,-29964.17, 26507.71]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](3,)","['Gr Liv Area','Bedroom AbvGr','Full Bath']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,4.667e+04
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,3
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(3)


In [16]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

knn_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("knn", KNeighborsRegressor())
])

param_grid_knn = {"knn__n_neighbors": range(1, 31)}

gscv_knn = GridSearchCV(knn_pipe, param_grid_knn, cv=10, scoring="neg_root_mean_squared_error")
gscv_knn.fit(X_train, y_train)

knn_model = gscv_knn.best_estimator_
print("best n_neighbors:", gscv_knn.best_params_)

best n_neighbors: {'knn__n_neighbors': 23}


In [17]:
sample_houses = X_train.sample(5, random_state=5544)

pd.DataFrame({
    "Gr Liv Area": sample_houses["Gr Liv Area"],
    "Bedroom AbvGr": sample_houses["Bedroom AbvGr"],
    "Full Bath": sample_houses["Full Bath"],
    "Linear prediction": linear_model.predict(sample_houses),
    "KNN prediction": knn_model.predict(sample_houses)
})

,Gr Liv Area,Bedroom AbvGr,Full Bath,Linear prediction,KNN prediction
1887,1187,2,1,154235.758338,124820.695652
1131,1080,3,1,111562.713497,133562.826087
1079,1358,2,2,201053.909260,202641.956522
333,874,3,1,87095.159345,112984.043478
2704,1073,2,2,167203.166865,129413.478261


In [18]:
from sklearn.ensemble import VotingRegressor

voting_model = VotingRegressor([
    ("linear", linear_model),
    ("knn", knn_model)
])
voting_model.fit(X=X_train, y=y_train)

pd.DataFrame({
    "Gr Liv Area": sample_houses["Gr Liv Area"],
    "Bedroom AbvGr": sample_houses["Bedroom AbvGr"],
    "Full Bath": sample_houses["Full Bath"],
    "Linear prediction": linear_model.predict(sample_houses),
    "KNN prediction": knn_model.predict(sample_houses),
    "Voting prediction": voting_model.predict(sample_houses)
})

,Gr Liv Area,Bedroom AbvGr,Full Bath,Linear prediction,KNN prediction,Voting prediction
1887,1187,2,1,154235.758338,124820.695652,139528.226995
1131,1080,3,1,111562.713497,133562.826087,122562.769792
1079,1358,2,2,201053.909260,202641.956522,201847.932891
333,874,3,1,87095.159345,112984.043478,100039.601412
2704,1073,2,2,167203.166865,129413.478261,148308.322563


In [19]:
from sklearn.ensemble import StackingRegressor

stacking_model = StackingRegressor([
    ("linear", linear_model),
    ("knn", knn_model)],
    final_estimator=LinearRegression()
)
stacking_model.fit(X=X_train, y=y_train)

pd.DataFrame({
    "Gr Liv Area": sample_houses["Gr Liv Area"],
    "Bedroom AbvGr": sample_houses["Bedroom AbvGr"],
    "Full Bath": sample_houses["Full Bath"],
    "Linear prediction": linear_model.predict(sample_houses),
    "KNN prediction": knn_model.predict(sample_houses),
    "Voting prediction": voting_model.predict(sample_houses),
    "Stacking prediction": stacking_model.predict(sample_houses)
})

,Gr Liv Area,Bedroom AbvGr,Full Bath,Linear prediction,KNN prediction,Voting prediction,Stacking prediction
1887,1187,2,1,154235.758338,124820.695652,139528.226995,128559.396286
1131,1080,3,1,111562.713497,133562.826087,122562.769792,128497.824582
1079,1358,2,2,201053.909260,202641.956522,201847.932891,203820.101909
333,874,3,1,87095.159345,112984.043478,100039.601412,106448.649741
2704,1073,2,2,167203.166865,129413.478261,148308.322563,134814.047759


In [20]:
from sklearn.model_selection import cross_val_score

for name, model in [("Linear", linear_model), ("KNN", knn_model), ("Voting", voting_model), ("Stacking", stacking_model)]:
    rmse = -cross_val_score(model, X=X_train, y=y_train, cv=10, scoring="neg_root_mean_squared_error").mean()
    print(f"{name} RMSE: {round(rmse, 2)}")

Linear RMSE: 51336.44
KNN RMSE: 47958.5
Voting RMSE: 48468.79
Stacking RMSE: 47977.75


Based on cross-validated RMSE, KNN (47958.5) and stacking (47977.75) are nearly identical and both clearly beat voting (48468.79) and linear regression (51336.44). Unlike the example where the ensemble beat both individual models, here KNN alone does just as well as stacking. Voting performs a bit worse than plain KNN because it averages a strong model with a weaker one equally, pulling the prediction toward the weaker linear model. Stacking avoids that by learning how much weight to give each model, letting it lean more heavily on KNN, so it keeps pace with KNN instead of getting dragged down.

## Voting in Classification

Suppose you have three models for predicting a categorical target:

- k-nearest neighbors
- logistic regression
- decision tree

How might you apply voting to create an ensemble model? Discuss conceptually with your partner. Create a concrete example to illustrate your method. Hint: There are two basic approaches to voting in classification; can you think of both?

Note: You can assume a binary target, but the general idea of voting applies to multiclass problems. However, if any of the candidate *models* are not naturally multiclass (like logistic regression) then you have to consider things like OvR and OvO as we have discussed.

There are two basic ways to do voting in classification. Hard voting just counts each model’s predicted class and goes with whichever class wins the most votes. If we imagine that KNN, logistic regression, and a decision tree are all predicting heart attack risk for one patient and KNN says “at risk”, logistic regression says “not at risk”, and the decision tree says “at risk.” Two votes for “at risk” beats one vote for “not at risk,” so hard voting predicts “at risk.”

Soft voting works differently. Instead of counting votes, it averages each model’s predicted probability of being at risk, then checks if that average is above 0.5. If KNN gives a 0.55 probability, logistic regression gives 0.40, and the decision tree gives 0.60, the average is about 0.52. This is above 0.5, so soft voting also predicts “at risk” here. Soft voting pays attention to how confident each model is, not just which side of 0.5 it landed on, so it won’t always agree with hard voting.

## Choose your own Classification

**Working together with your partner**, choose an interesting data set to fit *classification* models for, that is, a data set with a *categorical* target variable and several feature variables. Could be a data set that: we have used in class, we have used on an assignment, a data set from your project, you used in another course, or you just find interesting! You'll work together with your partner to develop *classification* models to predict the *categorical* target (so you need to decide on the data set and target variable together with your partner).

After you have decided on the data set and target variable:

- One partner should fit a k-nearest neighbors classification model
- One partner should fit another classification model like decision tree or logistic regression

If you're using an example from earlier in the class, you can use a model you have already come up with or you can try something new. Note: you do NOT have to agree on the features included in your models.

After each partner has their model, **work together to combine your code and create an ensemble model that uses *voting***.

For each of the three models (knn classification, other classification, voting ensemble)

1. Fit the model using the training data
1. Use the fitted model to predict the target for a few observations of your choice, and compare the predictions
1. Estimate the test macro precision, recall, and F1 score using cross-validation, and compare the results

Which model seems to work best?

**Hint:** [`VotingClassifier`](https://scikit-learn.org/1.5/modules/generated/sklearn.ensemble.VotingClassifier.html) works almost identically to `VotingRegressor`.


In [21]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

ha = pd.read_csv('/Users/brandonnewman/Desktop/GSB - 5544 - Fall 2026/data/heart_attack.csv')
ha = pd.get_dummies(ha, columns=["cp", "restecg"], drop_first=True)

X = ha.drop(columns="output")
y = ha["output"]

In [22]:
knn_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

param_grid_knn = {"knn__n_neighbors": range(1, 31)}
gscv_knn = GridSearchCV(knn_pipe, param_grid_knn, cv=10, scoring="roc_auc")
gscv_knn.fit(X, y)

knn_model = gscv_knn.best_estimator_
print("best n_neighbors:", gscv_knn.best_params_)

best n_neighbors: {'knn__n_neighbors': 24}


In [23]:
logreg_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=5000))
])

param_grid_logreg = {"logreg__C": np.logspace(-3, 3, 13)}
gscv_logreg = GridSearchCV(logreg_pipe, param_grid_logreg, cv=10, scoring="roc_auc")
gscv_logreg.fit(X, y)

logreg_model = gscv_logreg.best_estimator_
print("best C:", gscv_logreg.best_params_)

best C: {'logreg__C': np.float64(0.31622776601683794)}


In [24]:
sample_patients = X.sample(5, random_state=5544)

pd.DataFrame({
    "KNN prediction": knn_model.predict(sample_patients),
    "Logistic regression prediction": logreg_model.predict(sample_patients)
})

,KNN prediction,Logistic regression prediction
0,1,1
1,0,0
2,0,0
3,0,1
4,1,1


In [25]:
from sklearn.ensemble import VotingClassifier

voting_model = VotingClassifier([
    ("knn", knn_model),
    ("logreg", logreg_model)
], voting="soft")
voting_model.fit(X, y)

pd.DataFrame({
    "KNN prediction": knn_model.predict(sample_patients),
    "Logistic regression prediction": logreg_model.predict(sample_patients),
    "Voting prediction": voting_model.predict(sample_patients)
})

,KNN prediction,Logistic regression prediction,Voting prediction
0,1,1,1
1,0,0,0
2,0,0,0
3,0,1,0
4,1,1,1


In [26]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

for name, model in [("KNN", knn_model), ("Logistic Regression", logreg_model), ("Voting", voting_model)]:
    y_pred_cv = cross_val_predict(model, X, y, cv=10)
    precision = precision_score(y, y_pred_cv, average="macro")
    recall = recall_score(y, y_pred_cv, average="macro")
    f1 = f1_score(y, y_pred_cv, average="macro")
    print(f"{name}: precision={precision:.3f}, recall={recall:.3f}, f1={f1:.3f}")

KNN: precision=0.772, recall=0.771, f1=0.765
Logistic Regression: precision=0.769, recall=0.766, f1=0.767
Voting: precision=0.776, recall=0.775, f1=0.775


Voting comes out slightly ahead on all three metrics with precision 0.776, recall 0.775, and F1 0.775. KNN (0.772, 0.771, 0.765) and logistic regression (0.769, 0.766, 0.767). The margin is small, under a percentage point, but it holds across all three metrics. This matches the general idea behind ensembling, KNN and logistic regression make somewhat different mistakes on different patients, and combining them smooths some of those errors out.